In [1]:
import pandas as pd
import numpy as np
import time
import itertools
import re

# ============================================================
# CONFIG: file paths
# ============================================================
UNIT_DATA_PATH = "data/ELD_Test_System.xlsx"
DEMAND_DATA_PATH = "data/powerdemand.xlsx"
OUTPUT_FILE_PATH = "Merit Order (ED-UC-VPE-POZ).xlsx"

# ============================================================
# USER SETTINGS
# ============================================================
DEMAND_TOL = 1e-6
POZ_EPS = 1e-6

# Merit-order settings
MERIT_STEP_MW = 1.0
MERIT_MAX_ITERS = 100

# ============================================================
# Load Dataset
# ============================================================
data = pd.read_excel(UNIT_DATA_PATH)

a = data["a"].values.astype(float)
b = data["b"].values.astype(float)
c = data["c"].values.astype(float)

d = data["d"].values.astype(float)
e = data["e"].values.astype(float)

Minimum_Capacity = data["pmin"].values.astype(float)
Maximum_Capacity = data["pmax"].values.astype(float)

ramp_up = data["rampup"].values.astype(float)
ramp_down = data["rampdown"].values.astype(float)

Unit = data["Unit"].values

poz_raw = data["poz"].values if "poz" in data.columns else np.array([None] * len(Unit), dtype=object)

power_demand_data = pd.read_excel(DEMAND_DATA_PATH)
power_demand = power_demand_data["load"].values.astype(float)

# ============================================================
# POZ parsing utilities
# ============================================================
def parse_poz_cell(cell):
    if cell is None:
        return []
    if isinstance(cell, float) and np.isnan(cell):
        return []
    s = str(cell).strip()
    if s == "" or s.lower() in ["nan", "none", "null", "[]"]:
        return []

    nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
    vals = [float(x) for x in nums]
    if len(vals) < 2:
        return []

    intervals = []
    for i in range(0, len(vals) - 1, 2):
        lo, hi = vals[i], vals[i + 1]
        if hi < lo:
            lo, hi = hi, lo
        intervals.append((lo, hi))

    intervals.sort(key=lambda x: x[0])
    merged = []
    for lo, hi in intervals:
        if not merged or lo > merged[-1][1]:
            merged.append([lo, hi])
        else:
            merged[-1][1] = max(merged[-1][1], hi)
    return [(x[0], x[1]) for x in merged]

POZ_INTERVALS = [parse_poz_cell(cell) for cell in poz_raw]

def in_any_poz(p, intervals):
    for lo, hi in intervals:
        if lo <= p <= hi:
            return True
    return False

def project_out_of_poz_with_bounds(p, intervals, lb, ub, eps=POZ_EPS):
    p0 = float(np.clip(p, lb, ub))
    if not intervals:
        return p0
    if not in_any_poz(p0, intervals):
        return p0

    candidates = [lb, ub]
    for lo, hi in intervals:
        candidates += [lo - eps, hi + eps, lo, hi]

    feasible = []
    for x in candidates:
        x = float(np.clip(x, lb, ub))
        if not in_any_poz(x, intervals):
            feasible.append(x)

    if feasible:
        return min(feasible, key=lambda x: abs(x - p0))
    return p0

def apply_poz_vector_with_bounds(P, lb_vec, ub_vec, commitment):
    P = np.asarray(P, dtype=float).copy()
    lb_vec = np.asarray(lb_vec, dtype=float)
    ub_vec = np.asarray(ub_vec, dtype=float)
    commitment = np.asarray(commitment, dtype=int)

    for i in range(len(P)):
        if commitment[i] == 0:
            P[i] = 0.0
        else:
            P[i] = project_out_of_poz_with_bounds(P[i], POZ_INTERVALS[i], lb_vec[i], ub_vec[i])
    return P

# ============================================================
# POZ-aware effective UB helpers
# ============================================================
def highest_feasible_under_ub(lb, ub, intervals, eps=POZ_EPS):
    ub0 = float(ub)
    lb0 = float(lb)
    if ub0 < lb0:
        return lb0

    if not intervals:
        return ub0

    if not in_any_poz(ub0, intervals):
        return ub0

    candidates = [lb0]
    for (lo, hi) in intervals:
        candidates.append(lo - eps)

    feasible = []
    for x in candidates:
        x = float(np.clip(x, lb0, ub0))
        if not in_any_poz(x, intervals):
            feasible.append(x)

    if feasible:
        return max(feasible)
    return lb0

def compute_effective_ub(lb_vec, ub_vec, commitment):
    lb_vec = np.asarray(lb_vec, dtype=float)
    ub_vec = np.asarray(ub_vec, dtype=float)
    commitment = np.asarray(commitment, dtype=int)

    ub_eff = ub_vec.copy()
    for i in range(len(ub_eff)):
        if commitment[i] == 0:
            ub_eff[i] = 0.0
        else:
            ub_eff[i] = highest_feasible_under_ub(lb_vec[i], ub_vec[i], POZ_INTERVALS[i], eps=POZ_EPS)
            if ub_eff[i] < lb_vec[i]:
                ub_eff[i] = lb_vec[i]
    return ub_eff

# ============================================================
# CHECKERS
# ============================================================
def vpe_checker(P, commitment):
    P = np.asarray(P, dtype=float)
    commitment = np.asarray(commitment, dtype=int)
    vpe_term = np.abs(d * np.sin(e * (Minimum_Capacity - P)))
    vpe_term = np.where(commitment == 1, vpe_term, 0.0)
    active = np.where(commitment == 1, np.abs(vpe_term) > 1e-12, False)
    return vpe_term, float(np.sum(vpe_term)), active

def poz_checker(P, commitment):
    P = np.asarray(P, dtype=float)
    commitment = np.asarray(commitment, dtype=int)
    violation = np.zeros_like(P, dtype=bool)
    which_interval = [None] * len(P)

    for i in range(len(P)):
        if commitment[i] == 0:
            continue
        for (lo, hi) in POZ_INTERVALS[i]:
            if lo <= P[i] <= hi:
                violation[i] = True
                which_interval[i] = (lo, hi)
                break

    return violation, which_interval, int(np.sum(violation))

# ============================================================
# COST helpers
# ============================================================
def base_cost_per_unit(P_full):
    P_full = np.asarray(P_full, dtype=float)
    return (a + b * P_full + c * (P_full ** 2))

def incremental_cost(P_full):
    P_full = np.asarray(P_full, dtype=float)
    return (b + 2.0 * c * P_full)

# ============================================================
# Demand repair inside bounds
# ============================================================
def repair_to_meet_demand_with_bounds(P, demand, lb_vec, ub_vec, commitment, tol=DEMAND_TOL, max_iters=80):
    P = np.asarray(P, dtype=float).copy()
    lb_vec = np.asarray(lb_vec, dtype=float)
    ub_vec = np.asarray(ub_vec, dtype=float)
    commitment = np.asarray(commitment, dtype=int)

    P = np.clip(P, lb_vec, ub_vec)
    P = apply_poz_vector_with_bounds(P, lb_vec, ub_vec, commitment)

    for _ in range(max_iters):
        total = float(np.sum(P))
        if total >= demand - tol:
            break

        deficit = demand - total
        headroom = (ub_vec - P) * commitment
        headroom = np.maximum(headroom, 0.0)
        hsum = float(np.sum(headroom))
        if hsum <= tol:
            break

        inc = deficit * (headroom / hsum)
        P = P + inc
        P = np.clip(P, lb_vec, ub_vec)
        P = apply_poz_vector_with_bounds(P, lb_vec, ub_vec, commitment)

    total = float(np.sum(P))
    if total < demand - tol:
        deficit = demand - total
        headroom = (ub_vec - P) * commitment
        headroom = np.maximum(headroom, 0.0)
        if float(np.sum(headroom)) > tol:
            j = int(np.argmax(headroom))
            P[j] = min(P[j] + deficit, ub_vec[j])
            P = np.clip(P, lb_vec, ub_vec)
            P = apply_poz_vector_with_bounds(P, lb_vec, ub_vec, commitment)

    if float(np.sum(P)) < demand - tol:
        raise ValueError(f"Infeasible in bounds: Total {float(np.sum(P))} < Demand {demand}.")

    return P

def enforce_ramp_poz_and_demand(prev_P, curr_P, demand, commitment):
    prev = np.asarray(prev_P, dtype=float)
    cur = np.asarray(curr_P, dtype=float)
    commitment = np.asarray(commitment, dtype=int)

    lb = np.zeros_like(cur, dtype=float)
    ub = np.zeros_like(cur, dtype=float)

    for i in range(len(cur)):
        if commitment[i] == 0:
            lb[i] = 0.0
            ub[i] = 0.0
        else:
            lb[i] = max(Minimum_Capacity[i], prev[i] - ramp_down[i])
            ub[i] = min(Maximum_Capacity[i], prev[i] + ramp_up[i])
            if ub[i] < lb[i]:
                ub[i] = lb[i]

    ub_eff = compute_effective_ub(lb, ub, commitment)

    P = np.clip(cur, lb, ub_eff)
    P = apply_poz_vector_with_bounds(P, lb, ub_eff, commitment)
    P = repair_to_meet_demand_with_bounds(P, demand, lb, ub_eff, commitment)
    P = np.clip(P, lb, ub_eff)
    P = apply_poz_vector_with_bounds(P, lb, ub_eff, commitment)
    return P

# ============================================================
# UC Merit Order (based on IC at Pmin)
# ============================================================
def merit_order_uc(demand):
    n = len(Unit)
    order_key = b + 2.0 * c * Minimum_Capacity
    idx_sorted = np.argsort(order_key)

    commitment = np.zeros(n, dtype=int)
    pmin_sum = 0.0
    pmax_sum = 0.0

    for i in idx_sorted:
        commitment[i] = 1
        pmin_sum += Minimum_Capacity[i]
        pmax_sum += Maximum_Capacity[i]
        if pmin_sum <= demand <= pmax_sum:
            return commitment

    return np.ones(n, dtype=int)

def max_feasible_total_with_poz(prev_P, demand, commitment):
    n = len(commitment)
    if prev_P is None:
        lb = Minimum_Capacity * commitment
        ub = Maximum_Capacity * commitment
    else:
        prev_P = np.asarray(prev_P, dtype=float)
        lb = np.zeros(n, dtype=float)
        ub = np.zeros(n, dtype=float)
        for i in range(n):
            if commitment[i] == 0:
                lb[i] = 0.0
                ub[i] = 0.0
            else:
                lb[i] = max(Minimum_Capacity[i], prev_P[i] - ramp_down[i])
                ub[i] = min(Maximum_Capacity[i], prev_P[i] + ramp_up[i])
                if ub[i] < lb[i]:
                    ub[i] = lb[i]

    ub_eff = compute_effective_ub(lb, ub, commitment)
    return float(np.sum(ub_eff)), lb, ub, ub_eff

def escalate_uc_for_feasibility_poz(prev_P, demand, base_commitment):
    commitment = base_commitment.copy().astype(int)

    total_eff, _, _, _ = max_feasible_total_with_poz(prev_P, demand, commitment)
    if total_eff >= demand - DEMAND_TOL:
        return commitment

    off = np.where(commitment == 0)[0].tolist()
    if not off:
        return commitment

    scores = []
    for i in off:
        lb_i = max(Minimum_Capacity[i], 0.0 - ramp_down[i])
        ub_i = min(Maximum_Capacity[i], 0.0 + ramp_up[i])
        if ub_i < lb_i:
            gain = 0.0
        else:
            ub_eff_i = highest_feasible_under_ub(lb_i, ub_i, POZ_INTERVALS[i], eps=POZ_EPS)
            gain = max(0.0, ub_eff_i)
        scores.append((gain, i))

    scores.sort(reverse=True, key=lambda x: x[0])

    for gain, i in scores:
        if gain <= 0:
            continue
        commitment[i] = 1
        total_eff, _, _, _ = max_feasible_total_with_poz(prev_P, demand, commitment)
        if total_eff >= demand - DEMAND_TOL:
            return commitment

    return commitment

# ============================================================
# Merit Order Dispatch inside bounds + POZ-aware UB
# ============================================================
def merit_order_dispatch(demand, commitment, prev_P=None, step_mw=MERIT_STEP_MW):
    commitment = np.asarray(commitment, dtype=int)
    n = len(commitment)

    if prev_P is None:
        lb = Minimum_Capacity * commitment
        ub = Maximum_Capacity * commitment
    else:
        prev_P = np.asarray(prev_P, dtype=float)
        lb = np.zeros(n, dtype=float)
        ub = np.zeros(n, dtype=float)
        for i in range(n):
            if commitment[i] == 0:
                lb[i] = 0.0
                ub[i] = 0.0
            else:
                lb[i] = max(Minimum_Capacity[i], prev_P[i] - ramp_down[i])
                ub[i] = min(Maximum_Capacity[i], prev_P[i] + ramp_up[i])
                if ub[i] < lb[i]:
                    ub[i] = lb[i]

    ub_eff = compute_effective_ub(lb, ub, commitment)

    if float(np.sum(ub_eff)) < demand - DEMAND_TOL:
        raise ValueError(f"Infeasible: sum(ub_eff)={float(np.sum(ub_eff))} < demand={demand}")

    P = lb.copy()
    P = apply_poz_vector_with_bounds(P, lb, ub_eff, commitment)

    it = 0
    while float(np.sum(P)) < demand - DEMAND_TOL:
        it += 1
        if it > MERIT_MAX_ITERS:
            break

        headroom = (ub_eff - P) * commitment
        headroom = np.maximum(headroom, 0.0)
        if float(np.sum(headroom)) <= DEMAND_TOL:
            break

        candidates = np.where(headroom > DEMAND_TOL)[0]
        if candidates.size == 0:
            break

        ic = incremental_cost(P)
        j = int(candidates[np.argmin(ic[candidates])])

        deficit = demand - float(np.sum(P))
        delta = min(step_mw, headroom[j], deficit)
        if delta <= 0:
            break

        P[j] += delta
        P[j] = float(np.clip(P[j], lb[j], ub_eff[j]))
        P = apply_poz_vector_with_bounds(P, lb, ub_eff, commitment)

    P = repair_to_meet_demand_with_bounds(P, demand, lb, ub_eff, commitment)
    P = np.clip(P, lb, ub_eff)
    P = apply_poz_vector_with_bounds(P, lb, ub_eff, commitment)
    return P

# ============================================================
# Build sched row helper
# ============================================================
def build_sched_row(demand, uc, P, comp_time=None):
    P = np.asarray(P, dtype=float)
    uc = np.asarray(uc, dtype=int)

    base_vec = base_cost_per_unit(P)
    vpe_vec, _, _ = vpe_checker(P, uc)
    total_vec = base_vec + vpe_vec
    poz_v, poz_int, poz_n = poz_checker(P, uc)

    row = {"Demand": float(demand)}
    for i, u in enumerate(Unit):
        row[f"Unit {u} Status"] = int(uc[i])
        row[f"Unit {u} Power Produced"] = float(P[i])
        row[f"Unit {u} Base Cost"] = float(base_vec[i])
        row[f"Unit {u} VPE Ripple"] = float(vpe_vec[i])
        row[f"Unit {u} Total Cost"] = float(total_vec[i])
        row[f"Unit {u} POZViolation"] = bool(poz_v[i])
        row[f"Unit {u} POZInterval"] = str(poz_int[i]) if poz_int[i] is not None else ""

    total_power = float(np.sum(P))
    row["Total Power Produced"] = float(total_power)
    row["Feasible (P>=D)"] = bool(total_power >= float(demand) - DEMAND_TOL)
    row["POZ Violations Count"] = int(poz_n)
    if comp_time is not None:
        row["Computation Time"] = float(comp_time)
    return row

# ============================================================
# Main loop (RAW then ADJUSTED) + CHECKERS
# ============================================================
all_outputs = []
schedulling_raw = []
schedulling_adjusted = []

ramp_rates = []
generation_limits = []

last_adjusted = None

for t, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    demand = float(demand)

    uc = merit_order_uc(demand)

    if last_adjusted is not None:
        uc = escalate_uc_for_feasibility_poz(last_adjusted, demand, uc)

    # RAW
    try:
        P_raw = merit_order_dispatch(demand, uc, prev_P=last_adjusted, step_mw=MERIT_STEP_MW)
    except ValueError:
        uc = np.ones_like(uc, dtype=int)
        P_raw = merit_order_dispatch(demand, uc, prev_P=last_adjusted, step_mw=MERIT_STEP_MW)

    comp_time = time.time() - start_time
    schedulling_raw.append(build_sched_row(demand, uc, P_raw, comp_time))

    # ADJUSTED
    if last_adjusted is None:
        P_adj = P_raw.copy()
    else:
        P_adj = enforce_ramp_poz_and_demand(last_adjusted, P_raw, demand, uc)

    schedulling_adjusted.append(build_sched_row(demand, uc, P_adj, comp_time))

    # =========================
    # CHECKERS: Ramp + GenLimit + POZ (based on adjusted)
    # =========================
    poz_v, poz_int, poz_n = poz_checker(P_adj, uc)
    vpe_vec, _, vpe_active = vpe_checker(P_adj, uc)

    total_power = float(np.sum(P_adj))
    feasible = bool(total_power >= demand - DEMAND_TOL)

    # Ramp-rate checker (only if previous exists)
    if last_adjusted is not None:
        ramp = P_adj - last_adjusted
    else:
        ramp = np.zeros_like(P_adj)

    ramp_viol = np.zeros_like(P_adj, dtype=bool)
    for k in range(len(P_adj)):
        if uc[k] == 0:
            continue
        ramp_viol[k] = (ramp[k] > ramp_up[k] + 1e-9) or (ramp[k] < -ramp_down[k] - 1e-9)

    # Generation limits checker
    gen_viol = np.zeros_like(P_adj, dtype=bool)
    for k in range(len(P_adj)):
        if uc[k] == 0:
            gen_viol[k] = (abs(P_adj[k]) > 1e-9)
        else:
            gen_viol[k] = (P_adj[k] < Minimum_Capacity[k] - 1e-9) or (P_adj[k] > Maximum_Capacity[k] + 1e-9)

    rr = {
        "Demand": demand,
        "Total Power Produced": total_power,
        "Feasible (P>=D)": feasible,
        "VPE Total": float(np.sum(vpe_vec)),
        "Any VPE Active": bool(np.any(vpe_active)),
        "POZ Violations Count": int(poz_n),
        "Any POZ Violation": bool(poz_n > 0),
    }
    gl = rr.copy()

    for idx, u in enumerate(Unit):
        rr[f"Unit {u} RampRate"] = float(ramp[idx])
        rr[f"Unit {u} RampViolation"] = bool(ramp_viol[idx])
        rr[f"Unit {u} VPE"] = float(vpe_vec[idx])
        rr[f"Unit {u} POZViolation"] = bool(poz_v[idx])
        rr[f"Unit {u} POZInterval"] = str(poz_int[idx]) if poz_int[idx] is not None else ""

        gl[f"Unit {u} Power"] = float(P_adj[idx])
        gl[f"Unit {u} GenLimitViolation"] = bool(gen_viol[idx])
        gl[f"Unit {u} VPE"] = float(vpe_vec[idx])
        gl[f"Unit {u} POZViolation"] = bool(poz_v[idx])
        gl[f"Unit {u} POZInterval"] = str(poz_int[idx]) if poz_int[idx] is not None else ""

    ramp_rates.append(rr)
    generation_limits.append(gl)

    # store outputs (ADJUSTED is truth)
    base_vec = base_cost_per_unit(P_adj)
    total_vec = base_vec + vpe_vec

    output = pd.DataFrame({
        "Unit": Unit,
        "On/Off Status": uc,
        "Power Produced (MW)": P_adj,
        "Base Cost (Rp equiv)": base_vec,
        "VPE Ripple (Rp equiv)": vpe_vec,
        "Total Cost (Rp equiv)": total_vec,
        "POZ": [POZ_INTERVALS[i] for i in range(len(Unit))],
        "POZ Violation": poz_v,
        "POZ Violating Interval": [str(x) if x is not None else "" for x in poz_int],
    })

    all_outputs.append({
        "Demand": demand,
        "Output": output,
        "Total Power Produced": total_power,
        "Total Base Cost": float(np.sum(base_vec)),
        "Total VPE Ripple": float(np.sum(vpe_vec)),
        "Total Cost": float(np.sum(total_vec)),
        "Computation Time": float(comp_time),
        "Unit Commitment": uc.copy(),
    })

    last_adjusted = P_adj.copy()

# ============================================================
# SAVE
# ============================================================
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    summary_rows = []
    for item in all_outputs:
        summary_rows.append({
            "Demand": item["Demand"],
            "Total Power Produced": item["Total Power Produced"],
            "Feasible (P>=D)": bool(item["Total Power Produced"] >= item["Demand"] - DEMAND_TOL),
            "Total Base Cost": item["Total Base Cost"],
            "Total VPE Ripple": item["Total VPE Ripple"],
            "Total Cost": item["Total Cost"],
            "Computation Time": item["Computation Time"],
        })

    pd.DataFrame(schedulling_raw).to_excel(writer, sheet_name="Schedulling_Raw", index=False)
    pd.DataFrame(schedulling_adjusted).to_excel(writer, sheet_name="Schedulling_Adjusted", index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Summary", index=False)

    if ramp_rates:
        pd.DataFrame(ramp_rates).to_excel(writer, sheet_name="Ramp Rates", index=False)

    if generation_limits:
        pd.DataFrame(generation_limits).to_excel(writer, sheet_name="Generation Limits", index=False)

print(f"All results saved to '{OUTPUT_FILE_PATH}'")

All results saved to 'Merit Order (ED-UC-VPE-POZ).xlsx'
